In [ ]:
!pip install tensorflow

### Data Augmentation for Dataset Balancing

This cell performs **data augmentation** using TensorFlow to increase the number of images per class.

Use **ImageDataGenerator** to create augmented variations of images.

**Goal**

- Ensure each class has **400 images**.
- If a class has fewer images, augmented images are generated until it reaches the target count.

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
import numpy as np

dataset_dir = "clean_dataset"
TARGET_IMAGES = 400

# Image augmentation configuration
datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    fill_mode="nearest"
)

# Loop through each class folder
for class_name in os.listdir(dataset_dir):
    # Skip hidden files/folders like .DS_Store
    if class_name.startswith("."):
        continue

    class_path = os.path.join(dataset_dir, class_name)

    # Make sure it's a directory
    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    current_count = len(images)
    print(class_name, "original:", current_count)

    # Skip augmentation if class already has enough images
    if current_count >= TARGET_IMAGES:
        continue

    # Number of images needed to reach target
    needed = TARGET_IMAGES - current_count
    i = 0

    while i < needed:
        img_name = images[i % current_count]
        img_path = os.path.join(class_path, img_name)

        try:
            # Load image and convert to array
            img = load_img(img_path)
            x = img_to_array(img)
            x = np.expand_dims(x, axis=0)


            
            # Generate augmented image
            aug_iter = datagen.flow(
                x,
                batch_size=1,
                save_to_dir=class_path,
                save_prefix="aug",
                save_format="jpg"
            )

            next(aug_iter)
            i += 1

        except Exception as e:
            print("Error processing image:", img_path, e)
            i += 1  # Skip this image if there’s an error

    print(class_name, "augmented to", TARGET_IMAGES)

splitting of training and testing set 

In [1]:
import os, shutil, random

input_dir = "aug_dataset_with_unknown"
output_dir = "dataset_split"
train_ratio = 0.8

os.makedirs(output_dir, exist_ok=True)

for class_name in os.listdir(input_dir):
    class_path = os.path.join(input_dir, class_name)

    # Skip if not a directory
    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    split_idx = int(len(images) * train_ratio)
    train_images = images[:split_idx]
    test_images = images[split_idx:]

    # Create train/test folders
    train_class = os.path.join(output_dir, "train", class_name)
    test_class = os.path.join(output_dir, "test", class_name)
    os.makedirs(train_class, exist_ok=True)
    os.makedirs(test_class, exist_ok=True)

    # Copy images
    for img in train_images:
        shutil.copy(os.path.join(class_path, img), train_class)
    for img in test_images:
        shutil.copy(os.path.join(class_path, img), test_class)

split the validation set from training set 

In [ ]:
import os
import shutil
import random

train_dir = "dataset_split/train"  # your existing train folder
val_dir = "dataset_split/val"      # new validation folder
val_ratio = 0.2                     # 20% of train goes to validation

os.makedirs(val_dir, exist_ok=True)

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)

    if not os.path.isdir(class_path):
        continue  # skip hidden files like .DS_Store

    images = os.listdir(class_path)
    random.shuffle(images)

    num_val = int(len(images) * val_ratio)
    val_images = images[:num_val]

    val_class_path = os.path.join(val_dir, class_name)
    os.makedirs(val_class_path, exist_ok=True)

    for img in val_images:
        shutil.move(os.path.join(class_path, img), os.path.join(val_class_path, img))